# LangChain GenAI Chatbot — `init_chat_model()`

### Multi-turn conversation with LangChain message classes

This notebook converts the classroom whiteboard flow into a clean, practical implementation.

**Core flow**

> **LangChain → GenAI → Chat Model → Messages → Conversation Loop**

The implementation uses LangChain's unified `init_chat_model()` interface, `.env` configuration, and the message classes:

- `SystemMessage` — defines the assistant's behavior
- `HumanMessage` — represents the user's input
- `AIMessage` — represents the assistant's response

> **Goal:** Build a simple multi-turn chatbot while keeping model configuration outside the Python source code.


## 1. What the whiteboard is teaching

The class notes highlight four important ideas:

| Whiteboard idea | Notebook implementation |
|---|---|
| **LangChain → GenAI** | LangChain provides a common framework for working with chat models |
| **Unified module** | `init_chat_model()` gives one interface for different providers |
| **Specific modules** | Provider packages such as `langchain-openai`, `langchain-ollama`, `langchain-anthropic`, etc. |
| **`.env`** | Store configuration such as `MODEL_NAME` outside the source code |
| **Native message classes** | `SystemMessage`, `HumanMessage`, `AIMessage` |
| **`init_chat_model()`** | Creates the configured chat model |
| **Conversation history** | Keep messages in a list and send the list to the model on every turn |

### The key engineering idea

Instead of hard-coding a provider/model directly into the application, we can configure the model through an environment variable and keep the application code largely provider-agnostic.


## 2. Environment setup

Create a project environment first.

A typical project can look like:

```text
langchain-chatbot/
├── .env
├── .gitignore
└── chatbot.ipynb
```

Install the common packages:

```bash
pip install -U langchain langchain-core python-dotenv
```

Then install the **provider package** required by your model, for example:

```bash
pip install -U langchain-ollama
```

or:

```bash
pip install -U langchain-openai
```

### Important

The provider package is separate from the unified LangChain interface.  
This is why the classroom notes distinguish between a **unified interface** and **specific provider modules**.


## 3. Configure `.env`

Create a `.env` file in the same project directory.

### Example for a local Ollama model

```env
MODEL_NAME=ollama:qwen2.5:7b
```

### Example concept for another provider

```env
MODEL_NAME=<provider>:<model-name>
```

Do **not** commit secrets such as API keys to GitHub.

A useful `.gitignore` entry is:

```text
.env
.ipynb_checkpoints/
__pycache__/
```

> If your provider requires an API key, put the key in `.env` according to that provider's environment-variable convention.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = os.getenv("MODEL_NAME")

print("MODEL_NAME:", MODEL_NAME)


## 4. Import the LangChain message classes

A chat model does not have to receive plain strings only.

LangChain provides structured message objects:

### `SystemMessage`
Sets the behavior or instructions for the assistant.

### `HumanMessage`
Contains the user's message.

### `AIMessage`
Represents the assistant's previous response.

This structure makes conversation history explicit and easy to manage.


In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


## 5. Initialize the chat model

The classroom notes emphasize:

```python
init_chat_model()
```

This is the **unified model initialization interface**.

Instead of writing provider-specific initialization code throughout the application, we can initialize the configured chat model once and then use the same `invoke()` pattern.


In [ ]:
llm = init_chat_model(
    model=MODEL_NAME,
    temperature=0.1
)

print("Chat model initialized successfully.")


## 6. Create the conversation state

The first message is a `SystemMessage`.

Then, after every user turn:

1. Add a `HumanMessage`
2. Send the full message history to the model
3. Receive an `AIMessage`
4. Add that response to the history

This is what gives the chatbot **multi-turn context**.


In [ ]:
messages = [
    SystemMessage(content="You are a helpful AI Assistant!")
]

messages


## 7. Single-turn test

Before creating the interactive loop, test one question.

This makes debugging easier because we can verify the model connection independently.


In [ ]:
question = "What is Generative AI?"

messages.append(HumanMessage(content=question))

response = llm.invoke(messages)

messages.append(AIMessage(content=response.content))

print("Bot:", response.content)


## 8. Interactive chatbot — classroom version

The following cell closely follows the original Python script.

### Commands

Type any question to chat.

Use one of these to stop:

- `exit`
- `quit`
- `bye`


In [ ]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


load_dotenv()

llm = init_chat_model(
    model=os.getenv("MODEL_NAME"),
    temperature=0.1
)

messages = [
    SystemMessage(content="You are a helpful AI Assistant!")
]

while True:
    question = input("You: ").strip()

    if not question:
        continue

    if question in ("exit", "quit", "bye"):
        print("Goodbye!")
        break

    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)

    messages.append(AIMessage(content=response.content))

    print(f"Bot: {response.content}")


## 9. How the chatbot remembers the conversation

Suppose the user asks:

```text
You: My name is Ali.
Bot: Nice to meet you, Ali!

You: What is my name?
Bot: Your name is Ali.
```

The model can answer the second question because the application sends the previous messages again:

```text
SystemMessage
      ↓
HumanMessage
      ↓
AIMessage
      ↓
HumanMessage
      ↓
AIMessage
```

### Important distinction

The Python list `messages` is the application's **conversation history**.

The model itself is invoked with that history each time:

```python
response = llm.invoke(messages)
```

So the application is responsible for deciding what context to send.


## 10. Message flow

```text
                    ┌─────────────────────┐
                    │    SystemMessage    │
                    │ "You are helpful"   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │    HumanMessage     │
                    │    User question    │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │     Chat Model      │
                    │   llm.invoke(...)   │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │      AIMessage      │
                    │ Assistant response  │
                    └──────────┬──────────┘
                               │
                               ▼
                         messages.append()
                               │
                               └──────► Next turn
```


## 11. A cleaner reusable version

For application development, it is useful to put the chatbot logic inside a function.

The behavior is the same as the classroom script, but the code becomes easier to reuse later in:

- FastAPI
- a web application
- a CLI
- an AI agent workflow
- a course project


In [ ]:
def run_chatbot():
    messages = [
        SystemMessage(content="You are a helpful AI Assistant!")
    ]

    print("Chatbot started. Type 'exit' to stop.\n")

    while True:
        question = input("You: ").strip()

        if not question:
            continue

        if question.lower() in {"exit", "quit", "bye"}:
            print("Goodbye!")
            break

        messages.append(HumanMessage(content=question))

        response = llm.invoke(messages)

        messages.append(AIMessage(content=response.content))

        print("Bot:", response.content)
        print()


# Uncomment to start the chatbot:
# run_chatbot()


## 12. Why use `init_chat_model()`?

The main advantage is **a common programming interface**.

Conceptually:

```text
                    init_chat_model()
                           │
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
       OpenAI           Ollama          Anthropic
          │                │                │
          └────────────────┼────────────────┘
                           ▼
                    llm.invoke(...)
```

Your application can focus on the chat workflow rather than rewriting the entire application whenever the provider changes.

The exact provider/model configuration still depends on the provider package and its requirements.


## 13. `temperature=0.1` — simple explanation

```python
temperature=0.1
```

A low temperature generally encourages more consistent and less random responses.

For an educational chatbot or coding assistant, a low value can be a reasonable starting point.

Think of it simply as:

- **Low temperature** → more predictable
- **Higher temperature** → more variation/creativity

The exact behavior can vary by model/provider.


## 14. Common mistakes

### 1. `.env` is not loading

Check:

```python
from dotenv import load_dotenv
load_dotenv()
```

and verify that `.env` is in the project directory.

### 2. `MODEL_NAME` is `None`

Check:

```python
print(os.getenv("MODEL_NAME"))
```

### 3. Provider package is missing

Install the package required by your provider, such as:

```bash
pip install -U langchain-ollama
```

or:

```bash
pip install -U langchain-openai
```

### 4. API key or local model is unavailable

The model configuration can be correct while the provider itself is not configured or running. Check the provider's credentials, service, or local runtime.

### 5. Conversation becomes too large

The application sends `messages` on every turn. For long conversations, production systems usually need strategies such as:

- trimming old messages
- summarizing history
- storing/retrieving relevant context
- using a context window carefully


## 15. Mini exercise

Modify the chatbot so that:

1. The system prompt says the assistant is an **AI Engineering Tutor**.
2. The assistant answers in **simple English**.
3. Add a `/clear` command that resets the conversation.
4. Add a `/history` command that prints the current message history.
5. Add a `/model` command that displays `MODEL_NAME`.

### Challenge

Turn the notebook chatbot into a small **FastAPI** endpoint while keeping the same `llm.invoke(messages)` core idea.


# Key Takeaways

### Remember these five things

**1. `init_chat_model()`**  
A unified way to initialize a configured chat model.

**2. `.env`**  
Keeps model configuration and secrets outside the Python source code.

**3. Message classes**  
`SystemMessage`, `HumanMessage`, and `AIMessage` give structure to chat history.

**4. `llm.invoke(messages)`**  
Sends the current conversation context to the model.

**5. `messages`**  
The application maintains the conversation history and decides what context the model receives.

---

### Final mental model

```text
Configuration
    ↓
.env → MODEL_NAME
    ↓
init_chat_model()
    ↓
Chat Model (llm)
    ↓
SystemMessage + HumanMessage + AIMessage
    ↓
messages
    ↓
llm.invoke(messages)
    ↓
AI response
    ↓
append response
    ↓
next conversation turn
```

**LangChain is the framework.  
The chat model is the intelligence.  
The message history is the context.**
